## **Imports**
---

In [ ]:
import os
import sys
from pathlib import Path

USING_COLAB: bool = "google.colab" in sys.modules

if USING_COLAB:
    REPO_URL: str = "https://github.com/Johannes-Herz/tum-adlr-13"
    REPO_BRANCH: str = "feature/tactile-dataloader"
    REPO_DIR: str = "/content/tum-adlr-13"

    if not os.path.exists(REPO_DIR):
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
        !pip install -q torchmetrics pydantic-settings
    else:
        !git -C {REPO_DIR} fetch origin
        !git -C {REPO_DIR} checkout {REPO_BRANCH}
        !git -C {REPO_DIR} pull

    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")

    !pip install torchmetrics
    !pip install pydantic-settings

    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=True)

if not USING_COLAB:
    %load_ext autoreload
    %autoreload 2

from dotenv import load_dotenv

GOOGLE_DRIVE_PATH: Path = Path("/content/gdrive/MyDrive/tum-adlr-13")

if not load_dotenv():
    loaded_dotenv = load_dotenv(GOOGLE_DRIVE_PATH / ".env")
    print(f"Loaded .env from Drive: {loaded_dotenv}")

In [ ]:
import tactex
from tactex.reconstruction.data import OccupancyGridDatset, build_dataloader
from tactex.reconstruction.metrics import build_metrics, build_trackers
from tactex.reconstruction.training import train
from tactex.reconstruction.unet import UNet
from tactex.settings import get_settings
from tactex.timings import get_timings
from tactex.visualization import metrics as metrics_vis

print(tactex.__file__)

In [ ]:
from collections.abc import Callable
from pathlib import Path

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchmetrics import MetricCollection, MetricTracker

In [ ]:
import wandb
from wandb.sdk.lib.runid import generate_id

wandb.login()

## **Config**
---

In [ ]:
RESOLUTION: int = 64

if os.path.exists('/content/gdrive'):
    DATA_ROOT: Path = Path('/content/data')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    !cp -r /content/gdrive/MyDrive/tum-adlr-13/data/. /content/data/
    !ls /content/data/
else:
    DATA_ROOT: Path = get_settings().data_root
print("data root: ", DATA_ROOT)

WANDB_ENTITY: str = "tum-adlr-13"
WANDB_PROJECT: str = "037397390_test"

if USING_COLAB: 
    WANDB_DATA_DIR: Path = Path("/content/wandb")
else: 
    WANDB_DATA_DIR: Path = get_settings().project_root / "wandb"
WANDB_DATA_DIR.mkdir(parents=True, exist_ok=True)
print("wandb data dir: ", WANDB_DATA_DIR)

DEVICE: torch.device = torch.accelerator.current_accelerator(check_available=True)
if DEVICE is None:
    DEVICE = torch.device("cpu")
# DEVICE = torch.device("cpu")


print(DEVICE)

In [ ]:
!ls /content/data/datasets

In [ ]:
WANDB_RUN_ID: str | None = None
WANDB_RUN_NAME: str = "training"
print(f"wandb run name: {WANDB_RUN_NAME}")

## **Dataset**
---

In [ ]:
# dataset_directory: Path = DATA_ROOT / "datasets" / "circles" / f"res{str(RESOLUTION).zfill(3)}" / "v001"
dataset_directory: Path = DATA_ROOT / "full_grid" / "shapenet" / "03797390" / f"res{str(RESOLUTION).zfill(3)}" / "v001"

dataset: OccupancyGridDatset = OccupancyGridDatset(root=dataset_directory, in_memory=False)

generator: torch.Generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(dataset=dataset, lengths=[0.85, 0.15], generator=generator)
print("Train dataset length: ", len(train_dataset))
print("Val dataset length: ", len(val_dataset))

In [ ]:
print(train_dataset.indices[:10])
print(val_dataset.indices[:10])

In [ ]:
BATCH_SIZE: int = 64
NUM_WORKERS: int = 2

train_dataloader: DataLoader = build_dataloader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, resolution=RESOLUTION, max_rays=50, seed=None, num_workers=NUM_WORKERS, name="train.dataloader")
val_dataloader: DataLoader = build_dataloader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=True, resolution=RESOLUTION, max_rays=50, seed=100, num_workers=NUM_WORKERS, name="val.dataloader")

## **Training**
---

In [ ]:
widths: list[int] = [32, 64, 128, 256]
kernel_size: int = 3
learning_rate: float = 0.001

config = {
    "widths": widths,
    "kernel_size": kernel_size,
    "learning_rate": learning_rate,
}

In [ ]:
unet: UNet = UNet(in_channels=3, out_channels=1, widths=widths, kernel_size=kernel_size)
loss_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor] = nn.BCEWithLogitsLoss()
optimizer: optim.Adam = optim.Adam(params=unet.parameters(), lr=learning_rate)
metrics: MetricCollection = build_metrics(loss_fn=loss_fn, threshold=0.5)
train_tracker, val_tracker = build_trackers(metrics=metrics, device=DEVICE)
learning_rate_scheduler: optim.lr_scheduler.ReduceLROnPlateau =  optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer, 
    mode="min",
    factor=0.5,
    patience=4,
    min_lr=1e-6
)

if WANDB_RUN_ID is None:
    WANDB_RUN_ID = generate_id()
print("wandb run id:", WANDB_RUN_ID)

In [ ]:
EPOCHS: int = 1
with wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, id=WANDB_RUN_ID, name=WANDB_RUN_NAME, resume="allow", config=config, dir=WANDB_DATA_DIR) as run:
    wandb_callback = lambda results, _: run.log(results)
    train(model=unet, train_dataloader=train_dataloader, val_dataloader=val_dataloader, loss_fn=loss_fn, optimizer=optimizer, train_tracker=train_tracker, val_tracker=val_tracker, epochs=EPOCHS, device=DEVICE, callback=wandb_callback)

In [ ]:
print(get_timings())

### **Save model checkpoint**
---

In [ ]:
def make_persistent(metric_tracker: MetricTracker):
    for metric in metric_tracker:
        metric.persistent(True)

make_persistent(train_tracker)
make_persistent(val_tracker)

In [ ]:
epoch: int = train_tracker.n_steps
epoch_dir: Path = GOOGLE_DRIVE_PATH / "models" / WANDB_RUN_NAME / f"epoch_{epoch:06d}"
epoch_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving checkpoint for epoch '{epoch}' in '{epoch_dir}'")

In [ ]:
torch.save(
    {
        "model_state_dict": unet.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "widths": widths,
        "kernel_size": kernel_size,
        "learning_rate": learning_rate,
        "epoch": epoch,
        "train_tracker_state_dict": train_tracker.state_dict(),
        "train_tracker_n": len(train_tracker),
        "val_tracker_state_dict": val_tracker.state_dict(),
        "val_tracker_n": len(val_tracker),
        "scheduler_state_dict": learning_rate_scheduler.state_dict(),
    },
    epoch_dir / "unet.pt",
)

## **Visualization**
---

In [ ]:
metrics_vis.show(train_tracker=train_tracker, val_tracker=val_tracker)